# Nowcoder Interview Data — Cleaning & Supervised Classification

**MGS 4701 | Track 3 | Reproducible data-processing notebook**

**Purpose:** Reproduce the preliminary Nowcoder dataset from a prepared 944-post source and a 150-post user-reviewed labeled reference set. The reference labels are used to train a **TF-IDF character n-gram + linear SVM** classifier, predict unseen posts, preserve provided labels, and filter Nowcoder posts marked `include`.

**What this notebook does _not_ claim:** The 387 retained Nowcoder posts are **preliminary model-filtered candidates**, not 387 manually confirmed BA/DA interviews. The provided reference set has major platform-label imbalance; some supplied skill labels were AI-assisted. No new human labeling is performed here.

| Stage | Input / output | Records |
|---|---|---:|
| Prepared source data | `raw data.csv` | 944 (394 Nowcoder + 550 Reddit) |
| Reviewed reference labels | `Goldset.csv` | 150 (75 per platform) |
| Model prediction | Remaining posts | 794 |
| Retained Nowcoder posts | Existing labels + predicted `include` | Expected 387 (68 labeled + 319 predicted) |

*The expected counts describe the supplied version of the inputs. Running with revised labels or library versions may change the predictions.*

## 0. Files and reproducibility

Put **`Goldset.csv`** and **`raw data.csv`** next to this notebook (or in a `data/` directory). Install dependencies if needed:

```bash
python -m pip install numpy scikit-learn joblib
```

Run **Kernel → Restart & Run All**. Outputs are saved to `outputs/`.

The prepared source data already has the fields `record_id, platform, source_row, keyword, title, text, date, url`. This notebook verifies those records; it **does not reproduce the original web scraping or assert that all missing/truncated text was repaired**. The source `date` denotes the post date, not necessarily the interview date.

## 1. Load packages, define the codebook variables and text classifier

- **Stage A:** `role_match`, `actual_interview`, `usable_detail`.
- **Relevance:** `include` only if all three Stage A labels are `1`; `exclude` if any is `0`; otherwise `uncertain`.
- **Skills:** ten multi-label categories. `1` is reported as assessed; `0` means not reported in the available text; `U` means uncertain; `NA` is used for excluded or uncertain records.

**Important:** The supplied `role_match` labels are reused as provided. Some Nowcoder reference labels reflect a broader *"job named in title"* rule rather than strict BA/DA role relevance. The notebook does **not** quietly revise them.

In [ ]:
#!/usr/bin/env python3
"""Train supervised text classifiers on the 150 user-reviewed BA/DA coding examples.
No fixed keyword decision rules are used to assign relevance/skills. This is a
small-sample supervised baseline; provenance and cross-validation are explicit.
"""
import os, csv, json, math, collections, pathlib, datetime
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
import joblib

# Paths work when inputs are beside this notebook or inside a data/ folder.
BASE = pathlib.Path.cwd()

def find_input(filename):
    candidates = [
        BASE / filename,
        BASE / 'data' / filename,
        BASE.parent / filename,
        BASE.parent / 'data' / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f'Missing {filename}. Place it next to the notebook or under data/. '
        f'Searched: {[str(p) for p in candidates]}'
    )

HERE = BASE / 'outputs'
HERE.mkdir(parents=True, exist_ok=True)
SRC = find_input('source_944.csv')
REF = find_input('reference_150_user_reviewed.csv')
print('Inputs:', SRC, REF)
print('Outputs:', HERE.resolve())
SKILLS=['sql','programming','statistics','machine_learning','business_case','product_metrics','bi_visualization','behavioral','project_experience','ai_tool_assessment']
STAGE=['role_match','actual_interview','usable_detail']

def read(p):
 with open(p,'r',encoding='utf-8-sig',newline='') as f: return list(csv.DictReader(f))
def write(p,rows,keys=None):
 if not rows:return
 with open(p,'w',encoding='utf-8-sig',newline='') as f:
  w=csv.DictWriter(f,fieldnames=keys or list(rows[0]),extrasaction='ignore');w.writeheader();w.writerows(rows)
def txt(r):
 title=(r.get('title') or '').strip()
 body=(r.get('text') or '').strip()[:3000]
 return ' '.join([title]*3) +'\n'+body

def pipe():
 return Pipeline([
  ('tfidf',TfidfVectorizer(analyzer='char',ngram_range=(2,4),max_features=50000,min_df=2,sublinear_tf=True)),
  ('clf', LinearSVC(C=1.0,class_weight='balanced',random_state=4701))])

def fit(k,refs,idx=None,ignored=('NA','','U')):
 idx=list(range(len(refs))) if idx is None else idx
 ii=[i for i in idx if (refs[i].get(k) or '') not in ignored]
 y=[refs[i][k] for i in ii]
 if len(set(y))<2:return None,len(ii),dict(collections.Counter(y))
 model=pipe();model.fit([txt(refs[i]) for i in ii],y)
 return model,len(ii),dict(collections.Counter(y))

def row_pred(model,text_list):
 classes=list(map(str,model.classes_))
 p=list(map(str,model.predict(text_list)))
 score=np.asarray(model.decision_function(text_list))
 if score.ndim==1:
  # signed distance from separating hyperplane, divided by arbitrary constant; NOT probability
  margins=list(map(float,np.abs(score)))
 else:
  order=np.sort(score,axis=1)
  margins=list(map(float,order[:,-1]-order[:,-2]))
 return p,margins

## 2. Read and check the prepared data

Check record IDs, alignment of the 150 labels with the 944 sources, platform counts, missing URLs/text, and duplicate URLs. **Quality checks are diagnostics, not invented cleaning operations**: no post is removed merely because a field looks incomplete.

In [ ]:
sources=read(SRC); refs=read(REF)
assert len(sources)==944 and len(refs)==150
assert len({x['record_id'] for x in sources})==944
refmap={x['record_id']:x for x in refs}
assert len(refmap)==150 and all(k in {x['record_id'] for x in sources} for k in refmap)
ref_ids=set(refmap)
test=[s for s in sources if s['record_id'] not in ref_ids]
assert len(test)==794
print('Input:',len(sources),'Gold labels:',len(refs),'Unseen:',len(test), 'Gold status:',collections.Counter(r['relevance_status'] for r in refs),flush=True)

In [ ]:
from collections import Counter

print('Platform counts:', dict(Counter(r['platform'] for r in sources)))
print('Gold status by platform:', dict(Counter((r['platform'],r['relevance_status']) for r in refs)))
for label, rows in [('Prepared source', sources), ('Gold reference', refs)]:
    nonempty_urls = [r.get('url', '').strip() for r in rows if r.get('url', '').strip()]
    print(label, 'empty titles:', sum(not r.get('title','').strip() for r in rows),
          '| empty texts:', sum(not r.get('text','').strip() for r in rows),
          '| empty URLs:', len(rows)-len(nonempty_urls),
          '| duplicate nonempty URLs:', len(nonempty_urls)-len(set(nonempty_urls)))
nowcoder_source = [r for r in sources if r['platform'] == 'Nowcoder']
print('Nowcoder source rows with ellipsis markers (possible snippets):',
      sum('...' in (r.get('text') or '') or '……' in (r.get('text') or '')
          for r in nowcoder_source))

## 3. Evaluate the labeling method without copying Gold Set answers

Use **stratified 5-fold out-of-fold cross-validation**: for each fold, train on 4 folds and predict the held-out fold. Report accuracy and **macro F1**, not accuracy alone. Also compare against a **platform-only baseline**, which guesses `include` for Nowcoder and `exclude` for Reddit.

These checks estimate agreement with **the supplied labels**, not independent evidence that the labels are correct.

In [ ]:
# Hold out same fold across stage-A and status tasks. Never copy training labels into predictions during CV.
cross=StratifiedKFold(n_splits=5,shuffle=True,random_state=4701)
oof={k:['']*len(refs) for k in STAGE+['relevance_status']}
for train,test_idx in cross.split([txt(r) for r in refs], [r['relevance_status'] for r in refs]):
 for k in oof:
  keep=[int(i) for i in train if (refs[int(i)].get(k) or '') not in ('','NA') and not (k=='actual_interview' and refs[int(i)][k]=='U')]
  model=pipe(); model.fit([txt(refs[i]) for i in keep],[refs[i][k] for i in keep]);pred=model.predict([txt(refs[int(i)]) for i in test_idx]);
  for j,label in zip(test_idx,pred):oof[k][int(j)]=str(label)

def derive(r,a,d):
 if '0' in (str(r),str(a),str(d)):return 'exclude'
 if (str(r),str(a),str(d))==('1','1','1'):return 'include'
 return 'uncertain'

oof_derived=[derive(oof['role_match'][i],oof['actual_interview'][i],oof['usable_detail'][i]) for i in range(len(refs))]
crossval=[]
for k in oof:
 y=[r[k] for r in refs];p=oof[k];
 if k=='actual_interview':
  ix=[i for i in range(len(y)) if y[i] in ('0','1')]; y=[y[i] for i in ix];p=[p[i] for i in ix]
 labels=sorted(set(y)); crossval.append({'task':k,'n':len(y),'accuracy':round(accuracy_score(y,p),4),'macro_f1':round(f1_score(y,p,average='macro',zero_division=0),4), 'include_precision':None, 'include_recall':None,'include_f1':None, 'class_counts':json.dumps(dict(collections.Counter(y)),ensure_ascii=False),'notes':'5-fold out-of-fold predictions; provided labels may include AI-assisted recodes'})
 if k=='relevance_status':
  precision,recall,f,_=precision_recall_fscore_support(y,p,labels=['include'],zero_division=0)
  crossval[-1].update({'include_precision':round(float(precision[0]),4),'include_recall':round(float(recall[0]),4),'include_f1':round(float(f[0]),4)})
crossval.append({'task':'relevance_from_stageA','n':len(refs),'accuracy':round(accuracy_score([r['relevance_status'] for r in refs],oof_derived),4),'macro_f1':round(f1_score([r['relevance_status'] for r in refs],oof_derived,average='macro',zero_division=0),4),'class_counts':json.dumps(dict(collections.Counter(r['relevance_status'] for r in refs)),ensure_ascii=False),'notes':'5-fold CV; stage-A decisions derived without using gold relevance label in test fold'})

In [ ]:
gold_status = [r['relevance_status'] for r in refs]
platform_baseline = ['include' if r['platform']=='Nowcoder' else 'exclude' for r in refs]
print('Platform-only baseline accuracy:', round(accuracy_score(gold_status, platform_baseline), 4))
print('5-fold relevance accuracy:', [x['accuracy'] for x in crossval if x['task']=='relevance_status'][0])
print('5-fold relevance macro F1:', [x['macro_f1'] for x in crossval if x['task']=='relevance_status'][0])
print('Gold-set direct relevance confusion matrix (rows=gold, cols=predicted):')
classes = ['include','exclude','uncertain']
print('Order:',classes)
print(confusion_matrix(gold_status, oof['relevance_status'], labels=classes))
print('Gold-set include prevalence by platform:')
for platform in ('Nowcoder','Reddit'):
    vals=[r for r in refs if r['platform']==platform]
    print(platform, sum(r['relevance_status']=='include' for r in vals),'/',len(vals))

## 4. Train models on all 150 provided labels

Fit separate text models for the three Stage A fields and direct relevance. Skill models use only `include` rows with explicit `0`/`1` gold labels. A skill classifier is **not trained** if either class has fewer than four examples; its unseen-post output stays `U` rather than guessing. Skill-classifier cross-validation is also recorded where feasible.

Text representation: title repeated three times + first 3,000 characters of the post body, then character 2–4-gram TF-IDF and class-balanced linear SVM. This model **does not explicitly take `platform` as a feature**, but writing style and language can strongly encode platform.

In [ ]:
# Skill models: fit only explicitly available 0/1 labels in the gold-set included posts.
include_idx=[i for i,r in enumerate(refs) if r['relevance_status']=='include']
models={};ntrain={};counts={}
for k in STAGE+['relevance_status']:
 models[k],ntrain[k],counts[k]=fit(k,refs,ignored=('','NA') if k!='actual_interview' else ('','NA','U'))
for k in SKILLS:
 model,n,ct=fit(k,refs,idx=include_idx,ignored=('','NA','U'))
 counts[k]=ct;ntrain[k]=n
 # At least 4 examples of positive and negative for training, otherwise keep abstaining U.
 if ct.get('1',0)<4 or ct.get('0',0)<4:
  models[k]=None
  crossval.append({'task':k,'n':n,'accuracy':'','macro_f1':'','include_precision':'','include_recall':'','include_f1':'','class_counts':json.dumps(ct,ensure_ascii=False),'notes':'TOO FEW positive labels for learned skill classifier: mark new-post skill as U'})
  continue
 models[k]=model
 folds=min(5,min(ct.values()))
 cv=StratifiedKFold(n_splits=folds,shuffle=True,random_state=4701)
 idx=[i for i in include_idx if refs[i].get(k) in ('0','1')]
 prediction=['']*len(idx)
 for tr,te in cv.split([txt(refs[i]) for i in idx],[refs[i][k] for i in idx]):
  m=pipe();m.fit([txt(refs[idx[i]]) for i in tr],[refs[idx[i]][k] for i in tr]); pr=m.predict([txt(refs[idx[j]]) for j in te]);
  for j,v in zip(te,pr):prediction[int(j)]=str(v)
 true=[refs[i][k] for i in idx]
 p,r,f,_=precision_recall_fscore_support(true,prediction,labels=['1'],zero_division=0)
 crossval.append({'task':k,'n':len(true),'accuracy':round(accuracy_score(true,prediction),4),'macro_f1':round(f1_score(true,prediction,average='macro',zero_division=0),4),'include_precision':round(float(p[0]),4),'include_recall':round(float(r[0]),4),'include_f1':round(float(f[0]),4),'class_counts':json.dumps(ct,ensure_ascii=False),'notes':f'{folds}-fold CV on included training examples; no independent human validation'})

### 4a. Supplementary interview variables

Where a field has enough diverse training labels, fit optional models for reported rounds, assessment format and role level. When training information is insufficient, return `U`, `unknown` or `NA`, rather than generating unsupported details.

In [ ]:
# Supplemental labels learned only if enough class frequency.
for k in ['rounds_reported','assessment_format','role_level']:
 idx=include_idx;ct=collections.Counter(refs[i].get(k,'') for i in idx)
 keep=[i for i in idx if refs[i].get(k,'') not in ('','NA') and ct[refs[i][k]]>=3]
 if len(set(refs[i][k] for i in keep))>=2:
  models[k]=pipe().fit([txt(refs[i]) for i in keep],[refs[i][k] for i in keep]);ntrain[k]=len(keep)
 else:models[k]=None

## 5. Predict the remaining 794 posts

Predict Stage A and derive final inclusion from those Stage A predictions. Keep the direct relevance prediction as a **diagnostic comparison**. Linear SVM margins are **not probabilities**; low-margin and contradictory predictions enter the review queue. For interpretability, store three most similar reference IDs, but **do not copy their labels**.

In [ ]:
# Similar reference examples are a navigation aid, not validation of a prediction.
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2,4),
                             max_features=40000, min_df=2, sublinear_tf=True)
train_vectors = vectorizer.fit_transform([txt(r) for r in refs])
test_vectors = vectorizer.transform([txt(r) for r in test])
sims = cosine_similarity(test_vectors, train_vectors)

text_to_predict = [txt(r) for r in test]
pred_fields, model_margins = {}, {}

# Relevance decisions are needed for ALL 794 unseen posts.
for k in STAGE + ['relevance_status']:
    pred_fields[k], model_margins[k] = row_pred(models[k], text_to_predict)

# Skill predictions are meaningful ONLY for Stage-A-included posts.
include_positions = [i for i in range(len(test))
                     if derive(pred_fields['role_match'][i],
                               pred_fields['actual_interview'][i],
                               pred_fields['usable_detail'][i]) == 'include']
include_texts = [text_to_predict[i] for i in include_positions]
for k, model in models.items():
    if k in STAGE + ['relevance_status'] or model is None:
        continue
    if include_positions:
        subset_labels, subset_margins = row_pred(model, include_texts)
    else:
        subset_labels, subset_margins = [], []
    labels_full = ['NA'] * len(test)
    margins_full = [0.0] * len(test)
    for pos, label, margin in zip(include_positions, subset_labels, subset_margins):
        labels_full[pos] = label
        margins_full[pos] = margin
    pred_fields[k], model_margins[k] = labels_full, margins_full

print('Predicted Stage A',
      {k: dict(collections.Counter(pred_fields[k])) for k in STAGE + ['relevance_status']},
      flush=True)
print('Stage-A-included unseen posts:',len(include_positions))

### 5a. Create predictions and review reasons

The ten skill outputs are only provisional. `1` must be confirmed against the actual interview text before use in findings. The code preserves the original URL and raw title/text for traceability.

In [ ]:
res=[];issues=[]
for ix,s in enumerate(test):
 p={k:pred_fields[k][ix] for k in STAGE}
 derived=derive(p['role_match'],p['actual_interview'],p['usable_detail'])
 direct=pred_fields['relevance_status'][ix]
 include=(derived=='include')
 nearest=sorted(range(len(refs)),key=lambda j:sims[ix,j],reverse=True)[:3]
 neighbor_ids=', '.join(f"{refs[j]['record_id']}({sims[ix,j]:.2f})" for j in nearest)
 notes=[]
 if derived!=direct:notes.append('Stage A vs directly learned relevance disagree')
 # Margin units of SVM are not calibrated probabilities.
 margin=float(model_margins['relevance_status'][ix])
 if margin<0.32:notes.append('low relevance-classifier margin')
 if p['role_match']=='U':notes.append('role_match uncertain')
 if p['actual_interview']=='U':notes.append('interview occurrence uncertain')
 if derived=='uncertain':notes.append('uncertain under Stage A rules')
 if include and s['platform']=='Reddit':notes.append('Reddit include: manually review actual assessed skills')
 detail={}
 for k in SKILLS:
  if not include:detail[k]='NA'
  elif models[k] is None:detail[k]='U';notes.append(f'insufficient positive examples for {k}')
  else:
   pr=pred_fields[k][ix];sm=float(model_margins[k][ix]);
   # abstain when SVM margin near the boundary, particularly unreliable rare positives
   detail[k]= 'U' if sm<0.16 else pr
   if sm<0.16:notes.append(f'low margin {k}')
   # avoid blindly treating rare predicted positives as confirmed interview evidence
   if pr=='1' and crossval[-1] is not None:
    pass
 def field(k,when_include,else_val='NA'):
  return pred_fields.get(k,['']*len(test))[ix] if when_include and models.get(k) is not None else else_val
 rd=field('rounds_reported',include,'NA')
 fm=field('assessment_format',include,'NA')
 rl=field('role_level',include,'unknown')
 if include and rd not in ('0','1','U'):rd='U'
 if include and not fm:fm='unknown'
 if include and not rl:rl='unknown'
 body=s.get('text','') or ''
 completeness=('snippet' if '...' in body or '……' in body else 'unknown') if s['platform']=='Nowcoder' else 'unknown'
 priority='HIGH' if derived!=direct or derived=='uncertain' or (include and s['platform']=='Reddit') else ('MEDIUM' if notes else 'LOW')
 row={**s,'label_origin':'supervised_model_prediction','coder_id':'ML-trained-on-150; NOT human', 'role_match':p['role_match'], 'actual_interview':p['actual_interview'],'usable_detail':p['usable_detail'], 'relevance_status':derived, **detail,
      'rounds_reported':rd,'round_count':'','assessment_format':fm,'role_level':rl,'company_type':'unknown','text_completeness':completeness,
      'evidence_text':'','decision_note':'Predicted by supervised text classifier; original title+text require human evidence check','model_direct_relevance':direct,'model_relevance_margin':round(margin,4),'nearest_gold_examples':neighbor_ids,
      'review_priority':priority,'review_reason':'; '.join(dict.fromkeys(notes)) if notes else 'Model prediction; verify before findings'}
 res.append(row)
 if priority!='LOW':issues.append({k:row.get(k,'') for k in ['record_id','platform','title','url','relevance_status','role_match','actual_interview','usable_detail','model_direct_relevance','model_relevance_margin','nearest_gold_examples','review_priority','review_reason']})

## 6. Preserve the 150 reference labels and combine with predictions

For the 150 training records, retain the **provided labels**, marking them as `provided_150_training_label`. The other 794 are marked `supervised_model_prediction`. Never treat the preserved training labels as held-out model predictions or calculate accuracy on them.

In [ ]:
# Training set is NOT a model prediction: preserve user-supplied labels, do not call it held-out accuracy.
trainrows=[]
for s in sources:
 if s['record_id'] not in refmap:continue
 g=refmap[s['record_id']]
 trainrows.append({**s,'label_origin':'provided_150_training_label', 'coder_id':g.get('coder_id','provided label'),**{k:g.get(k,'') for k in STAGE+['relevance_status']+SKILLS+['rounds_reported','round_count','assessment_format','role_level','company_type','text_completeness','evidence_text','decision_note']},
  'model_direct_relevance':'NOT PREDICTED - TRAINING','model_relevance_margin':'','nearest_gold_examples':'','review_priority':'TRAINING_LABEL','review_reason':'User-provided labeling standard. May include AI-assisted skills; not independently tested.'})
all_rows=sorted(trainrows+res,key=lambda r:(r['platform'],r['record_id']))
assert len(all_rows)==944 and len(set(r['record_id'] for r in all_rows))==944
cols=['record_id','platform','source_row','keyword','title','text','date','url','label_origin','coder_id']+STAGE+['relevance_status']+SKILLS+['rounds_reported','round_count','assessment_format','role_level','company_type','text_completeness','evidence_text','decision_note','model_direct_relevance','model_relevance_margin','nearest_gold_examples','review_priority','review_reason']
write(HERE/'all_944_learned_labels.csv',all_rows,cols)
write(HERE/'predicted_794_only.csv',res,cols)
write(HERE/'provided_150_training_labels.csv',trainrows,cols)
write(HERE/'review_queue.csv',issues)
write(HERE/'cross_validation_metrics.csv',crossval)

### 6a. Save the trained model and cross-validation diagnostics

The model, full 944-row CSV, 794 predictions, validation metrics and review queue are exported automatically to `outputs/`.

In [ ]:
# Save model pipeline and OOF diagnostics for reproducibility.
joblib.dump({'models':models,'vectorizer':vectorizer,'code_version':'v1.0_150supervised','skills':SKILLS,'trained_on':150,'training_record_ids':[r['record_id'] for r in refs]},HERE/'supervised_model_150.joblib',compress=3)
oof_rows=[]
for i,r in enumerate(refs):
 oof_rows.append({'record_id':r['record_id'],'platform':r['platform'],'actual_gold_status':r['relevance_status'], 'oof_direct_status':oof['relevance_status'][i], 'oof_derived_status':oof_derived[i],**{f'oof_{k}':oof[k][i] for k in STAGE}})
write(HERE/'oof_reference_150.csv',oof_rows)
print('Result statuses on 794',dict(collections.Counter(r['relevance_status'] for r in res)))
print('By platform',dict(collections.Counter((r['platform'],r['relevance_status']) for r in res)))
print('Review count',len(issues))
print('TOTAL',len(all_rows))
print('CV', [x for x in crossval if x['task'] in ['relevance_status','role_match','business_case','ai_tool_assessment']])

## 7. Extract the Nowcoder dataset (the group's preliminary "cleaned" output)

Apply both conditions:

1. `platform == "Nowcoder"`
2. `relevance_status == "include"`

This reproduces the previously shared **387-post preliminary dataset** from the supplied inputs. It **does not** remove non-BA/DA jobs that may have been included by broad title rules or by the model. Keep the entire raw Nowcoder subset as a separate file.

In [ ]:
nowcoder_raw = [r for r in sources if r['platform']=='Nowcoder']
nowcoder_include = [r for r in all_rows
                    if r['platform']=='Nowcoder' and r['relevance_status']=='include']
nowcoder_review = [r for r in nowcoder_include if r['label_origin']=='supervised_model_prediction']
assert len(nowcoder_raw)==394, f'Unexpected Nowcoder source count: {len(nowcoder_raw)}'
if len(nowcoder_include) != 387:
    print('NOTE: The number of Nowcoder includes has changed. Check input labels and library versions.')

# Export the prepared raw subset and the preliminary filtered dataset.
write(HERE/'nowcoder_raw_394.csv', nowcoder_raw,
      ['record_id','platform','source_row','keyword','title','text','date','url'])
write(HERE/'nowcoder_cleaned_387_PRELIMINARY.csv', nowcoder_include, cols)

print('Prepared raw Nowcoder:', len(nowcoder_raw))
print('Preliminary Nowcoder includes:', len(nowcoder_include))
print('By label origin:', dict(Counter(r['label_origin'] for r in nowcoder_include)))
print('Saved:', (HERE/'nowcoder_cleaned_387_PRELIMINARY.csv').resolve())

## 8. Final quality-control checklist and reporting language

**Do not silently convert model predictions into manually confirmed interviews.** Create a separate, clearly marked review list for Nowcoder predictions that are potential non-BA/DA jobs or have incomplete text. These flags help humans review; they are **not automatic exclusions** and do not modify the 387-row output.

For the final **BA/DA-only** analysis dataset, an actual human should verify job relevance, actual interviews, readable details and skill evidence, then recalculate the final `include` count. The older 40-post Pilot did **not** train this model and is **not** a substitute for independent validation.

In [ ]:
import re

ROLE_SCOPE_FLAG = re.compile(r'后端|算法|开发岗|运维|游戏运营|产品经理|测试工程师|前端开发|数据工程师')
review_rows = []
for r in nowcoder_include:
    flags=[]
    if r['label_origin']=='supervised_model_prediction':
        flags.append('predicted_include_not_manually_confirmed')
    if ROLE_SCOPE_FLAG.search(r.get('title','')):
        flags.append('possible_role_outside_BA_DA_CHECK_TITLE')
    if r.get('text_completeness')=='snippet':
        flags.append('possible_truncated_text')
    if r.get('review_priority') in ('HIGH','MEDIUM'):
        flags.append('model_review_priority_'+r.get('review_priority'))
    if flags:
        review_rows.append({**r,'QC_flags':'; '.join(flags)})

write(HERE/'nowcoder_review_queue.csv',review_rows,cols+['QC_flags'])
print('Flagged for possible human review:',len(review_rows),'of',len(nowcoder_include))
print('Files available for GitHub:')
for path in sorted(HERE.glob('*')):
    print(' -',path.name)
print('\nSuggested report wording:')
print('From 394 prepared Nowcoder candidate posts, 387 were provisionally retained '
      '(68 with provided labels and 319 with model predictions). '
      'Because the classifier shows platform-related bias, the retained set requires '
      'manual verification before being treated as final BA/DA interview data.')